In [1]:
# Load data
gdf = gpd.read_file('./berlin_wheelchair_nodes.geojson')

print(f"Total nodes: {len(gdf)}")
print(gdf['wheelchair'].value_counts())

# Filter and map labels
valid_classes = ['yes', 'designated', 'limited', 'no']
gdf = gdf[gdf['wheelchair'].isin(valid_classes)].copy()

# Combine 'designated' into 'yes'
gdf['wheelchair'] = gdf['wheelchair'].replace('designated', 'yes')

# Encode labels
label_encoder = LabelEncoder()
gdf['label'] = label_encoder.fit_transform(gdf['wheelchair'])
class_names = label_encoder.classes_

print("\nClass mapping:")
for i, name in enumerate(class_names):
    print(f"{i}: {name}")

print(f"\nFiltered nodes: {len(gdf)}")
print(gdf['wheelchair'].value_counts())

NameError: name 'gpd' is not defined

## 2. Load Data and Preprocess Labels

We load the Berlin wheelchair nodes GeoJSON.
Target mapping:
- `yes`, `designated` -> `yes` (Class 0)
- `limited` -> `limited` (Class 1)
- `no` -> `no` (Class 2)

In [2]:
# Install necessary packages if not already installed
# !pip install geopandas torch torch-geometric scikit-learn libpysal

import geopandas as gpd
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

print("Libraries imported.")

ModuleNotFoundError: No module named 'torch'

# Train GNN for Wheelchair Accessibility Prediction

This notebook trains a Graph Neural Network (GNN) to classify OSM nodes into `yes` (accessible), `no` (not accessible), or `limited` (partially accessible).
It uses spatial proximity to build a graph and node tags as features.

## 1. Install and Import Dependencies

## 3. Feature Engineering

We extract features from other OSM tags. We'll focus on tags that are likely relevant to accessibility (e.g., `amenity`, `shop`, `surface`, `smoothness`, `highway`).
We'll use One-Hot Encoding for categorical features.

In [ ]:
# Identify potential feature columns
# We exclude 'wheelchair', 'id', 'geometry', and other metadata
exclude_cols = ['wheelchair', 'id', 'geometry', 'label', '@id', 'timestamp', 'version', 'changeset', 'user', 'uid']
feature_cols = [c for c in gdf.columns if c not in exclude_cols]

# Select top K most frequent columns if there are too many
# Or manually select relevant ones if known. For now, let's pick top 20 most populated columns
top_k_cols = gdf[feature_cols].notna().sum().sort_values(ascending=False).head(20).index.tolist()
print(f"Selected feature columns: {top_k_cols}")

# Extract features
X_df = gdf[top_k_cols].copy()

# Handle missing values (fill with 'unknown' or similar)
X_df = X_df.fillna('missing')

# One-hot encode
X_encoded = pd.get_dummies(X_df, dummy_na=True)
print(f"Feature matrix shape: {X_encoded.shape}")

# Convert to tensor
x = torch.tensor(X_encoded.values, dtype=torch.float)
y = torch.tensor(gdf['label'].values, dtype=torch.long)

## 4. Construct Spatial Graph

We build a K-Nearest Neighbors (KNN) graph based on spatial coordinates (latitude, longitude).
This defines the "neighborhood" for the GNN. Nodes close to each other will exchange information.

In [ ]:
# Extract coordinates
coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))

# Build KNN graph
k = 10  # Number of neighbors
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='ball_tree').fit(coords)
distances, indices = nbrs.kneighbors(coords)

# Create edge index for PyG
# indices contains [self, neighbor_1, ..., neighbor_k]
# We need source -> target pairs
source_nodes = []
target_nodes = []

for i in range(len(coords)):
    for j in range(1, k+1): # Skip self (index 0)
        source_nodes.append(i)
        target_nodes.append(indices[i][j])
        # Add bidirectional edges? Usually yes for spatial graphs
        source_nodes.append(indices[i][j])
        target_nodes.append(i)

edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
print(f"Graph created with {edge_index.shape[1]} edges.")

## 5. Create PyG Data Object and Split

We wrap everything into a `torch_geometric.data.Data` object and create train/val/test masks.

In [ ]:
data = Data(x=x, edge_index=edge_index, y=y)

# Create masks
num_nodes = data.num_nodes
indices = np.arange(num_nodes)
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2, random_state=42, stratify=y[train_idx])

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"Train nodes: {train_mask.sum().item()}")
print(f"Val nodes: {val_mask.sum().item()}")
print(f"Test nodes: {test_mask.sum().item()}")

## 6. Define GNN Model

We use a Graph Convolutional Network (GCN).
Input -> GCNConv -> ReLU -> Dropout -> GCNConv -> LogSoftmax

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, num_features, num_classes):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(num_features, 64)
        self.conv2 = GCNConv(64, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

model = GCN(num_features=data.num_features, num_classes=3)
print(model)

## 7. Train the Model

We train for 200 epochs using Adam optimizer and Negative Log Likelihood Loss (NLLLoss).

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)
    
    # Calculate accuracy for train, val, test
    train_acc = (pred[data.train_mask] == data.y[data.train_mask]).sum().item() / data.train_mask.sum().item()
    val_acc = (pred[data.val_mask] == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()
    test_acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    
    return train_acc, val_acc, test_acc, pred

losses = []
for epoch in range(200):
    loss = train()
    losses.append(loss)
    if epoch % 20 == 0:
        train_acc, val_acc, test_acc, _ = test()
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

plt.plot(losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 8. Evaluate Performance

Detailed classification report on the test set.

In [ ]:
_, _, _, pred = test()

y_true = data.y[data.test_mask].cpu().numpy()
y_pred = pred[data.test_mask].cpu().numpy()

print(classification_report(y_true, y_pred, target_names=class_names))